# 02. End-to-End Graph Flow 테스트

이 노트북은 사용자 질문 1개가 입력되면 LangGraph의 전체 흐름을 거쳐  
최종 응답이 생성되기까지의 과정을 보여준다.

## 흐름
```
사용자 질문
  └── planner_node  ← PlannerService가 query_type, steps 결정
       └── route_by_query_type  ← DB / RAG / GENERAL 분기
            ├── DB 경로: generate_sql → validate_sql → execute_sql → summarize_db
            │    └── DB_THEN_RAG: retrieve_rag → final_answer
            │    └── 기타: final_answer
            ├── RAG 경로: retrieve_rag → final_answer
            └── GENERAL 경로: general_answer → final_answer
```

## 실행 전 준비사항
- `config/config.yaml` 설정 확인
- Ollama 서버 또는 OpenAI API 준비
- DB 연결 또는 벡터스토어 필요 (DB/RAG 경로 테스트 시)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"프로젝트 루트: {project_root}")

## 그래프 구조 시각화

In [ ]:
from src.db_to_llm.stream.graph.builder import build_graph

graph = build_graph()

# Mermaid 다이어그램 출력 시도
try:
    mermaid_png = graph.get_graph().draw_mermaid_png()
    from IPython.display import Image, display
    display(Image(mermaid_png))
except Exception as e:
    print(f"다이어그램 렌더링 실패 (선택 사항): {e}")

# 노드 목록
print("\n그래프 노드 목록:")
for node in graph.get_graph().nodes:
    print(f"  - {node}")

## 테스트 1: GENERAL 질문 (외부 의존성 없음)

In [ ]:
import json
from src.db_to_llm.stream.graph.runner import run_graph

config_path = str(project_root / "config" / "config.yaml")

question_general = "파이썬에서 제너레이터(generator)란 무엇인가요?"
print(f"질문: {question_general}")
print("=" * 60)

result = run_graph(question=question_general, config_path=config_path)

print(f"query_type   : {result.get('query_type')}")
print(f"planner_plan :\n{json.dumps(result.get('planner_result', {}), ensure_ascii=False, indent=2)}")
print(f"\n최종 답변:")
print(result.get('final_answer', '(없음)'))

## 테스트 2: trace_logs 및 errors 확인

In [ ]:
print("\n=== 실행 추적 로그 ===")
for log in result.get("trace_logs", []):
    print(f"  {log}")

if result.get("errors"):
    print("\n=== 오류 목록 ===")
    for err in result["errors"]:
        print(f"  [오류] {err}")

## 테스트 3: DB_ONLY 질문 (DB 연결 필요)

In [ ]:
# DB 연결 설정이 되어있는 경우에만 실행
# config.yaml의 database 섹션에 올바른 연결 정보가 있어야 함

question_db = "현재 테이블 목록을 조회해줘"
print(f"질문: {question_db}")
print("=" * 60)

result_db = run_graph(question=question_db, config_path=config_path)

print(f"query_type      : {result_db.get('query_type')}")
print(f"generated_sql   : {result_db.get('generated_sql', '(없음)')}")
print(f"validated_sql   : {result_db.get('validated_sql', '(없음)')}")
print(f"sql_valid       : {result_db.get('sql_validation_passed')}")
print(f"db_row_count    : {result_db.get('db_row_count')}")
print(f"db_summary:\n{result_db.get('db_summary', '(없음)')}")
print(f"\n최종 답변:\n{result_db.get('final_answer', '(없음)')}")

## 테스트 4: RAG_ONLY 질문 (벡터스토어 필요)

In [ ]:
# notebooks/ingest/01_build_vectorstore.ipynb를 먼저 실행하여
# ChromaDB에 문서가 업로드되어 있어야 한다

question_rag = "오류 코드 E2001이 발생했을 때 어떻게 처리해야 하나요?"
print(f"질문: {question_rag}")
print("=" * 60)

result_rag = run_graph(question=question_rag, config_path=config_path)

print(f"query_type       : {result_rag.get('query_type')}")
print(f"검색된 컨텍스트 수: {len(result_rag.get('retrieved_contexts', []))}")

for i, ctx in enumerate(result_rag.get("retrieved_contexts", [])[:3], 1):
    print(f"\n[컨텍스트 {i}]")
    print(f"  source: {ctx.get('source', 'N/A')}")
    snippet = ctx.get('text', '')[:200]
    print(f"  text  : {snippet}...")

print(f"\n최종 답변:\n{result_rag.get('final_answer', '(없음)')}")

## API 서버 연동 테스트 (선택 사항)

In [ ]:
# API 서버 실행 명령:
# uvicorn src.db_to_llm.stream.api.app:app --reload --port 8000

# 아래는 서버가 실행 중일 때만 동작한다
try:
    import requests
    response = requests.post(
        "http://localhost:8000/api/query",
        json={"question": "파이썬이란 무엇인가요?"},
        timeout=60
    )
    if response.status_code == 200:
        data = response.json()
        print(f"query_type: {data.get('query_type')}")
        print(f"최종 답변 :\n{data.get('final_answer', '(없음)')}")
    else:
        print(f"서버 오류: {response.status_code} - {response.text}")
except Exception as e:
    print(f"API 서버 연결 실패 (서버가 실행 중인지 확인): {e}")